# Divisão e conquista: teorema mestre e par de pontos mais próximo — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As medições e os gráficos ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- medir o custo $\Theta(n^2)$ da **força bruta** para o par de pontos mais próximo;
- aplicar o **teorema mestre** a recorrências $T(n) = a\,T(n/b) + \Theta(n^d)$ e conferir a resposta numericamente;
- resolver o par mais próximo **na reta** por divisão e conquista, identificando o único candidato que cruza a divisa;
- implementar e testar a **combinação na faixa** do algoritmo no plano, verificando o limite de 7 vizinhos;
- comparar empiricamente $\Theta(n^2)$, $\Theta(n\log^2 n)$ e $\Theta(n\log n)$.

**Exemplo condutor** (o mesmo dos slides): $A(1,5)$, $B(2,9)$, $C(4,5)$, $D(5,8)$, $E(6,8)$, $F(8,8)$,
$G(10,2)$, $H(11,9)$. O par mais próximo é $\{D,E\}$, com distância 1.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True)
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída dos programas deste tutorial."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            return float(tok.split('=')[1])
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

# Entrada do exemplo condutor: n e depois os pontos, um por linha.
EXEMPLO = '8\n1 5\n2 9\n4 5\n5 8\n6 8\n8 8\n10 2\n11 9\n'

## 1. O problema motivador: força bruta

A solução óbvia calcula a distância de **todos os pares**: $\binom{n}{2} = n(n-1)/2$ distâncias,
ou seja, $\Theta(n^2)$. O programa abaixo é o mesmo trecho mostrado nos slides, com a distância
instrumentada para contar quantas vezes é chamada.

Rode a célula e confira: no exemplo condutor são $\binom{8}{2} = 28$ distâncias e a resposta é $d = 1$.

**Fonte:** Levitin (2012), §3.3; Cormen et al. (2009), §33.4.

In [ ]:
%%writefile src/forca_bruta.c
/* PAR MAIS PROXIMO POR FORCA BRUTA: examina todos os n(n-1)/2 pares.
   Uso: ./forca_bruta n semente   (pontos aleatorios)  ou  ./forca_bruta < entrada
   Levitin (2012), sec. 3.3. */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

typedef struct { double x, y; } Ponto;

static long comparacoes = 0;

static double dist( Ponto p, Ponto q) {
   ++comparacoes;
   double dx = p.x - q.x, dy = p.y - q.y;
   return sqrt( dx*dx + dy*dy);
}

static double forcaBruta( Ponto P[], int n) {
   double d = INFINITY;
   for (int i = 0; i < n; ++i)
      for (int j = i+1; j < n; ++j) {
         double dij = dist( P[i], P[j]);
         if (dij < d) d = dij;
      }
   return d;
}

int main( int argc, char *argv[]) {
   int n;  Ponto *P;
   if (argc >= 3) {                               /* n pontos uniformes em [0,10^6)^2 */
      n = atoi( argv[1]);  srand( atoi( argv[2]));
      P = malloc( n * sizeof (Ponto));
      for (int i = 0; i < n; ++i) {
         P[i].x = rand() / (RAND_MAX + 1.0) * 1e6;
         P[i].y = rand() / (RAND_MAX + 1.0) * 1e6;
      }
   } else {                                       /* le n e os pontos da entrada */
      if (scanf( "%d", &n) != 1) return 1;
      P = malloc( n * sizeof (Ponto));
      for (int i = 0; i < n; ++i)
         if (scanf( "%lf %lf", &P[i].x, &P[i].y) != 2) return 1;
   }
   clock_t t0 = clock();
   double d = forcaBruta( P, n);
   double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;
   printf( "n=%d d=%.6f comparacoes=%ld tempo=%.4f\n", n, d, comparacoes, seg);
   free( P);
   return 0;
}

In [ ]:
# Exemplo: força bruta no exemplo condutor e em entradas aleatórias crescentes.
saida = rodar('forca_bruta.c', entrada=EXEMPLO)
assert campo(saida, 'd') == 1.0 and campo(saida, 'comparacoes') == 28

# Dobrar n deve (aproximadamente) quadruplicar o tempo: assinatura de Theta(n^2).
tempos_fb = {}
for n in [2000, 4000, 8000, 16000]:
    s = rodar('forca_bruta.c', args=(n, 42), mostrar=False)
    tempos_fb[n] = campo(s, 'tempo')
    print(f'n={n:6d}  comparacoes={int(campo(s, "comparacoes")):>12,d}  tempo={tempos_fb[n]:.4f} s')
ns = sorted(tempos_fb)
for a, b in zip(ns, ns[1:]):
    print(f'T({b})/T({a}) = {tempos_fb[b] / max(tempos_fb[a], 1e-9):.2f}   (esperado ~4)')

In [ ]:
# Exercício 1: estimando o tempo da força bruta
# TODO: a partir do maior tempo medido acima, estime (em segundos) quanto a força
# bruta levaria para n = 10**6 pontos. Use que o tempo cresce com n**2.

def estimar_tempo(n_medido, tempo_medido, n_alvo):
    # TODO: implemente aqui (uma linha basta)
    pass

# Teste automático (com números redondos, independente da sua máquina)
assert abs(estimar_tempo(1000, 0.5, 2000) - 2.0) < 1e-9, 'Dobrar n deve quadruplicar o tempo'
assert abs(estimar_tempo(10**4, 0.1, 10**6) - 1000.0) < 1e-6, 'Verifique sua implementação'
print(f'Estimativa para n = 10^6: {estimar_tempo(16000, tempos_fb[16000], 10**6) / 60:.1f} min')

## 2. Divisão e conquista e o teorema mestre

Um algoritmo de divisão e conquista que gera $a$ subproblemas de tamanho $n/b$ e gasta
$f(n)$ para dividir e combinar tem custo $T(n) = a\,T(n/b) + f(n)$. Com $f(n) \in \Theta(n^d)$,
o **teorema mestre** (Levitin, 2012, cap. 5) diz:

| condição | quem domina a árvore | $T(n)$ |
|---|---|---|
| $a < b^d$ | a raiz | $\Theta(n^d)$ |
| $a = b^d$ | todos os níveis empatam | $\Theta(n^d \log n)$ |
| $a > b^d$ | as folhas | $\Theta(n^{\log_b a})$ |

A célula abaixo **calcula** $T(n)$ pela própria recorrência (com $T(1) = 1$) e divide pelo
resultado previsto. Se o teorema estiver certo, a razão tende a uma **constante**.

**Fonte:** Levitin (2012), cap. 5; Cormen et al. (2009), §4.5.

In [ ]:
# Exemplo: conferindo o teorema mestre numericamente.
import math
from functools import lru_cache
import matplotlib.pyplot as plt

def T(a, b, f):
    """Devolve a função n -> T(n) da recorrência T(n) = a T(n//b) + f(n), T(1) = 1."""
    @lru_cache(maxsize=None)
    def t(n):
        return 1 if n <= 1 else a * t(n // b) + f(n)
    return t

casos = {
    # nome: (a, b, f(n), previsão do teorema)
    'busca binária  T(n/2)+1':     (1, 2, lambda n: 1,     lambda n: math.log2(n)),
    'mergesort     2T(n/2)+n':     (2, 2, lambda n: n,     lambda n: n * math.log2(n)),
    'ingênua       4T(n/2)+n':     (4, 2, lambda n: n,     lambda n: n ** 2),
    'Karatsuba     3T(n/2)+n':     (3, 2, lambda n: n,     lambda n: n ** math.log2(3)),
    'raiz domina   2T(n/2)+n^2':   (2, 2, lambda n: n * n, lambda n: n ** 2),
}

ns = [2 ** k for k in range(4, 21)]
plt.figure(figsize=(8, 4))
for nome, (a, b, f, prev) in casos.items():
    t = T(a, b, f)
    razoes = [t(n) / prev(n) for n in ns]
    plt.plot(ns, razoes, marker='.', label=nome)
    print(f'{nome}:  T(n)/previsão em n=2^20 = {razoes[-1]:.3f}')
plt.xscale('log', base=2)
plt.xlabel('n'); plt.ylabel('T(n) / previsão')
plt.title('A razão se estabiliza: a previsão do teorema mestre está certa')
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Exercício 2: o teorema mestre como função
# TODO: implemente teorema_mestre(a, b, d), que recebe os parâmetros de
# T(n) = a T(n/b) + Theta(n^d) e devolve uma tupla (expoente, potencia_do_log)
# tal que T(n) = Theta(n^expoente * log(n)^potencia_do_log).
# Exemplos: mergesort (2, 2, 1) -> (1, 1);  busca binária (1, 2, 0) -> (0, 1).
# Dica: compare a com b**d; no caso das folhas o expoente é math.log(a, b).
import math

def teorema_mestre(a, b, d):
    # TODO: implemente aqui
    pass

# Teste automático
def _perto(r, esperado):
    return r is not None and all(abs(x - y) < 1e-9 for x, y in zip(r, esperado))
assert _perto(teorema_mestre(2, 2, 1), (1, 1)), 'mergesort: Theta(n log n)'
assert _perto(teorema_mestre(1, 2, 0), (0, 1)), 'busca binária: Theta(log n)'
assert _perto(teorema_mestre(4, 2, 1), (2, 0)), 'multiplicação ingênua: Theta(n^2)'
assert _perto(teorema_mestre(3, 2, 1), (math.log2(3), 0)), 'Karatsuba: Theta(n^log2(3))'
assert _perto(teorema_mestre(7, 2, 2), (math.log2(7), 0)), 'Strassen: Theta(n^log2(7))'
assert _perto(teorema_mestre(2, 2, 2), (2, 0)), 'raiz domina: Theta(n^2)'
print('Todos os testes passaram!')

## 3. Aquecimento: par mais próximo na reta

Com os números **ordenados**, divida na mediana: $S_1$ fica com a metade esquerda e $S_2$ com a
direita. Resolva cada metade e obtenha $\delta_1$ e $\delta_2$. Um par que cruza a divisa só
interessa se for mais curto que ambos, e o único candidato é **o último de $S_1$ com o primeiro
de $S_2$**. A recorrência é $T(n) = 2T(n/2) + \Theta(1)$, portanto $\Theta(n)$ depois da
ordenação.

A célula de exemplo resolve o problema do jeito "óbvio" (ordenar e comparar vizinhos). No
exercício você escreve a versão por divisão e conquista, que é o esqueleto do algoritmo no plano.

**Fonte:** Levitin (2012), §5.5; Shamos e Hoey (1975), §2.

In [ ]:
%%writefile src/reta_ordena.c
/* PAR MAIS PROXIMO NA RETA: ordenar e comparar vizinhos consecutivos. */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

static int cmp( const void *a, const void *b) {
   double x = *(const double *) a, y = *(const double *) b;
   return (x > y) - (x < y);
}

int main( void) {
   int n;
   if (scanf( "%d", &n) != 1) return 1;
   double *x = malloc( n * sizeof (double));
   for (int i = 0; i < n; ++i) if (scanf( "%lf", &x[i]) != 1) return 1;
   qsort( x, n, sizeof (double), cmp);            /* Theta(n log n) */
   double d = INFINITY;
   for (int i = 0; i + 1 < n; ++i)                 /* Theta(n) */
      if (x[i+1] - x[i] < d) d = x[i+1] - x[i];
   printf( "d=%.6f\n", d);
   free( x);
   return 0;
}

In [ ]:
# Exemplo: as abscissas do exemplo condutor, projetadas na reta.
RETA = '8\n1 2 4 5 6 8 10 11\n'
saida = rodar('reta_ordena.c', entrada=RETA)
assert campo(saida, 'd') == 1.0
_ = rodar('reta_ordena.c', entrada='5\n9.5 -3 7 20 2\n')   # esperado: d=2.500000

In [ ]:
%%writefile src/reta_dc.c
/* Exercicio 3: PAR MAIS PROXIMO NA RETA POR DIVISAO E CONQUISTA.
   TODO: complete a funcao pmp1d(). O vetor x[lo..hi] ja' esta' ordenado. */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

static int cmp( const void *a, const void *b) {
   double x = *(const double *) a, y = *(const double *) b;
   return (x > y) - (x < y);
}

/* Devolve a menor distancia entre dois elementos de x[lo..hi]. */
static double pmp1d( double x[], int lo, int hi) {
   if (hi - lo < 1) return INFINITY;              /* um so' ponto: nao ha' par */
   int meio = (lo + hi) / 2;                       /* S1 = x[lo..meio], S2 = x[meio+1..hi] */
   /* TODO: CONQUISTAR - calcule d1 e d2 recursivamente */
   /* TODO: COMBINAR  - considere tambem o unico par que cruza a divisa */
   return 0;                                        /* TODO: devolva a resposta correta */
}

int main( void) {
   int n;
   if (scanf( "%d", &n) != 1) return 1;
   double *x = malloc( n * sizeof (double));
   for (int i = 0; i < n; ++i) if (scanf( "%lf", &x[i]) != 1) return 1;
   qsort( x, n, sizeof (double), cmp);
   printf( "d=%.6f\n", pmp1d( x, 0, n - 1));
   free( x);
   return 0;
}

In [ ]:
# Teste automático do Exercício 3: compara pmp1d com a versão "ordena e compara vizinhos".
import random
assert campo(rodar('reta_dc.c', entrada=RETA), 'd') == 1.0, 'Exemplo condutor: d deve ser 1'
random.seed(0)
for _ in range(30):
    n = random.randint(2, 40)
    xs = ' '.join(f'{random.uniform(-100, 100):.3f}' for _ in range(n))
    ent = f'{n}\n{xs}\n'
    esperado = campo(rodar('reta_ordena.c', entrada=ent, mostrar=False), 'd')
    obtido = campo(rodar('reta_dc.c', entrada=ent, mostrar=False), 'd')
    assert abs(esperado - obtido) < 1e-9, f'Verifique sua implementação: entrada {ent!r}'
print('Todos os testes passaram!')

## 4. Par mais próximo no plano em $O(n \log n)$

O programa abaixo é o algoritmo completo da aula (Cormen et al., 2009, §33.4):

1. **pré-ordena** uma vez: `X` por $x$ e `Y` por $y$;
2. **divide** pela mediana de `X` e separa `Y` em `YL` e `YR` **sem perder a ordem por $y$**,
   em tempo linear (o inverso do `merge` do mergesort);
3. **conquista** as duas metades e faz $\delta = \min\{\delta_L, \delta_R\}$;
4. **combina** examinando a faixa $|x - x_m| < \delta$, em que cada ponto é comparado apenas com os
   seguintes cujo $y$ difere em menos de $\delta$ (no máximo 7).

Um detalhe de implementação: com $x$ repetidos, "estar à esquerda da mediana" fica ambíguo.
O campo `id` define uma ordem lexicográfica $(x, y, \text{id})$ sem empates, e com ela `YL` contém
exatamente os pontos de `X[0..meio-1]`.

Recorrência: $T(n) = 2T(n/2) + \Theta(n)$, que é o caso $a = b^d$ e dá $\Theta(n \log n)$.

In [ ]:
%%writefile src/par.c
/* PAR DE PONTOS MAIS PROXIMO: forca bruta x divisao e conquista.
   Uso: ./par fb|dc n semente    (gera n pontos aleatorios)
        ./par fb|dc              (le n e os pontos da entrada padrao)
   Baseado em Cormen et al. (2009), sec. 33.4, e Levitin (2012), sec. 5.5. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

typedef struct { double x, y; int id; } Ponto;   /* id desempata pontos iguais */

static long comparacoes = 0;   /* quantas distancias foram calculadas */

static double dist( Ponto p, Ponto q) {
   ++comparacoes;
   double dx = p.x - q.x, dy = p.y - q.y;
   return sqrt( dx*dx + dy*dy);
}

/* ---------- forca bruta: todos os n(n-1)/2 pares ---------- */
static double forcaBruta( Ponto P[], int n) {
   double d = INFINITY;
   for (int i = 0; i < n; ++i)
      for (int j = i+1; j < n; ++j) {
         double dij = dist( P[i], P[j]);
         if (dij < d) d = dij;
      }
   return d;
}

/* ---------- divisao e conquista ---------- */
/* ordem lexicografica (x, y, id): nunca ha' empate */
static int antes( Ponto p, Ponto q) {
   if (p.x != q.x) return p.x < q.x;
   if (p.y != q.y) return p.y < q.y;
   return p.id < q.id;
}
static int porX( const void *a, const void *b) {
   const Ponto *p = a, *q = b;
   return antes( *p, *q) ? -1 : antes( *q, *p);
}
static int porY( const void *a, const void *b) {
   const Ponto *p = a, *q = b;
   return (p->y > q->y) - (p->y < q->y);
}

/* X[0..n-1]: pontos ordenados por x.
   Y[0..n-1]: os mesmos pontos ordenados por y. */
static double pmp( Ponto X[], Ponto Y[], int n) {
   if (n <= 3) return forcaBruta( X, n);          /* base */

   int meio = n / 2;
   Ponto divisor = X[meio];                      /* reta vertical x = divisor.x */

   /* DIVIDIR: separa Y em YL e YR preservando a ordem por y (tempo linear).
      Vai para a esquerda quem vem antes do divisor na ordem de X. */
   Ponto *YL = malloc( meio * sizeof (Ponto));
   Ponto *YR = malloc( (n - meio) * sizeof (Ponto));
   int l = 0, r = 0;
   for (int i = 0; i < n; ++i) {
      if (antes( Y[i], divisor)) YL[l++] = Y[i];
      else                       YR[r++] = Y[i];
   }

   /* CONQUISTAR: resolve as duas metades recursivamente */
   double dL = pmp( X, YL, meio);
   double dR = pmp( X + meio, YR, n - meio);
   double d = dL < dR ? dL : dR;
   free( YL);  free( YR);

   /* COMBINAR: faixa F com os pontos a menos de d da reta, ja' em ordem de y */
   Ponto *F = malloc( n * sizeof (Ponto));
   int f = 0;
   for (int i = 0; i < n; ++i)
      if (fabs( Y[i].x - divisor.x) < d) F[f++] = Y[i];

   for (int i = 0; i < f; ++i)
      for (int j = i+1; j < f && F[j].y - F[i].y < d; ++j) {  /* no maximo 7 */
         double dij = dist( F[i], F[j]);
         if (dij < d) d = dij;
      }
   free( F);
   return d;
}

static double divisaoConquista( Ponto P[], int n) {
   Ponto *X = malloc( n * sizeof (Ponto));
   Ponto *Y = malloc( n * sizeof (Ponto));
   memcpy( X, P, n * sizeof (Ponto));
   memcpy( Y, P, n * sizeof (Ponto));
   qsort( X, n, sizeof (Ponto), porX);            /* pre-ordenacao: uma vez so' */
   qsort( Y, n, sizeof (Ponto), porY);
   double d = pmp( X, Y, n);
   free( X);  free( Y);
   return d;
}

int main( int argc, char *argv[]) {
   if (argc < 2) { fprintf( stderr, "uso: %s fb|dc [n semente]\n", argv[0]); return 1; }
   int n;  Ponto *P;
   if (argc >= 4) {
      n = atoi( argv[2]);  srand( atoi( argv[3]));
      P = malloc( n * sizeof (Ponto));
      for (int i = 0; i < n; ++i) {
         P[i].x = rand() / (RAND_MAX + 1.0) * 1e6;
         P[i].y = rand() / (RAND_MAX + 1.0) * 1e6;
         P[i].id = i;
      }
   } else {
      if (scanf( "%d", &n) != 1) return 1;
      P = malloc( n * sizeof (Ponto));
      for (int i = 0; i < n; ++i) {
         if (scanf( "%lf %lf", &P[i].x, &P[i].y) != 2) return 1;
         P[i].id = i;
      }
   }
   clock_t t0 = clock();
   double d = strcmp( argv[1], "fb") == 0 ? forcaBruta( P, n) : divisaoConquista( P, n);
   double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;
   printf( "n=%d d=%.6f comparacoes=%ld tempo=%.4f\n", n, d, comparacoes, seg);
   free( P);
   return 0;
}

In [ ]:
# Exemplo: o exemplo condutor e uma verificação cruzada contra a força bruta.
saida = rodar('par.c', args=('dc',), entrada=EXEMPLO)
assert campo(saida, 'd') == 1.0

for semente in range(1, 21):
    for n in [2, 3, 4, 5, 8, 50, 777]:
        fb = campo(rodar('par.c', args=('fb', n, semente), mostrar=False), 'd')
        dc = campo(rodar('par.c', args=('dc', n, semente), mostrar=False), 'd')
        assert fb == dc, (n, semente, fb, dc)
print('Divisão e conquista concorda com a força bruta em 140 instâncias aleatórias.')

In [ ]:
# Visualização: tempo da força bruta x divisão e conquista (escala log-log).
import matplotlib.pyplot as plt

ns_fb = [1000, 2000, 4000, 8000, 16000, 32000]
ns_dc = [1000, 4000, 16000, 64000, 256000, 1000000]
t_fb = [campo(rodar('par.c', args=('fb', n, 7), mostrar=False), 'tempo') for n in ns_fb]
t_dc = [campo(rodar('par.c', args=('dc', n, 7), mostrar=False), 'tempo') for n in ns_dc]
c_dc = [campo(rodar('par.c', args=('dc', n, 7), mostrar=False), 'comparacoes') for n in ns_dc]

plt.figure(figsize=(8, 4))
plt.loglog(ns_fb, t_fb, 'o-', label='força bruta  $\\Theta(n^2)$')
plt.loglog(ns_dc, t_dc, 's-', label='divisão e conquista  $\\Theta(n\\log n)$')
plt.xlabel('n'); plt.ylabel('tempo (s)'); plt.grid(alpha=0.3, which='both')
plt.title('Par de pontos mais próximo'); plt.legend()
plt.show()

for n, c in zip(ns_dc, c_dc):
    print(f'n={n:8d}: divisão e conquista calculou {int(c):>9,d} distâncias '
          f'(a força bruta calcularia {n*(n-1)//2:>15,d})')

In [ ]:
%%writefile src/faixa.c
/* Exercicio 4: A COMBINACAO NA FAIXA.
   F[0..f-1] contem os pontos da faixa JA' ORDENADOS POR y; d e' o delta atual.
   TODO: complete combinaFaixa() para
     (a) devolver o novo delta (o menor entre d e as distancias examinadas);
     (b) guardar em *maxViz o maior numero de vizinhos examinados por um mesmo ponto.
   Examine F[j] (j > i) somente enquanto F[j].y - F[i].y < d (d ja' atualizado). */
#include <stdio.h>
#include <math.h>

typedef struct { double x, y; } Ponto;

static double dist( Ponto p, Ponto q) {
   return sqrt( (p.x-q.x)*(p.x-q.x) + (p.y-q.y)*(p.y-q.y));
}

static double combinaFaixa( Ponto F[], int f, double d, int *maxViz) {
   *maxViz = 0;
   /* TODO: implemente aqui */
   return d;
}

int main( void) {
   /* faixa do exemplo condutor (delta = 2): C, D, E */
   Ponto F1[] = { {4,5}, {5,8}, {6,8} };
   int m1;  double d1 = combinaFaixa( F1, 3, 2.0, &m1);
   /* uma faixa mais cheia, largura 2 (delta = 2) */
   Ponto F2[] = { {0,0}, {1.5,0.2}, {0.3,1.1}, {1.2,1.9}, {0.1,2.6}, {1.7,3.0}, {0.8,3.9} };
   int m2;  double d2 = combinaFaixa( F2, 7, 2.0, &m2);
   printf( "d1=%.6f max1=%d d2=%.6f max2=%d\n", d1, m1, d2, m2);
   return 0;
}

In [ ]:
# Teste automático do Exercício 4
saida = rodar('faixa.c')
assert abs(campo(saida, 'd1') - 1.0) < 1e-6 and campo(saida, 'max1') == 1, 'Exemplo condutor: D-E, 1 vizinho'
assert abs(campo(saida, 'd2') - 1.140175) < 1e-6, 'Faixa 2: menor distância errada'
assert campo(saida, 'max2') == 2, 'Faixa 2: contagem de vizinhos errada'
assert campo(saida, 'max2') <= 7
print('Todos os testes passaram!')

## Desafio Final

Nos slides vimos duas versões do algoritmo no plano:

- **versão 1**: em cada chamada, monta a faixa a partir de `X` e a **ordena por $y$** com `qsort`.
  A recorrência é $T(n) = 2T(n/2) + \Theta(n \log n)$, que cai na lacuna do teorema mestre, e a
  extensão do caso 2 dá $\Theta(n \log^2 n)$;
- **versão 2** (`src/par.c`): pré-ordena `Y` uma única vez, com $\Theta(n \log n)$.

Implemente a **versão 1** em `src/par_v1.c`, com a mesma interface de `par.c` (`./par_v1 dc n semente`
imprime `n=... d=... comparacoes=... tempo=...`). Depois:

1. confira que ela devolve o mesmo `d` que `par.c` em várias sementes;
2. meça as duas versões para $n = 2^{14}, 2^{15}, \dots, 2^{20}$;
3. faça o gráfico de `tempo / (n log2 n)` para as duas. Qual das curvas cresce, e por quê?
   Relacione a resposta com a recorrência de cada versão.

In [ ]:
# Desafio: versão Theta(n log^2 n) x versão Theta(n log n)
# TODO: resolva o desafio abaixo
# Sugestão: comece copiando src/par.c para src/par_v1.c e troque o parâmetro Y
# pela ordenação da faixa (qsort com porY) dentro de pmp().

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §3.3, cap. 5 e §5.5.
- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. §4.5 (método mestre) e §33.4 (par mais próximo).
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Pearson, 2006. §5.4.
- SHAMOS, M. I.; HOEY, D. Closest-point problems. *16th Annual Symposium on Foundations of Computer Science*, 1975, p. 151–162. DOI 10.1109/SFCS.1975.8.
- FEOFILOFF, P. *Análise de Algoritmos* (IME-USP): <https://www.ime.usp.br/~pf/analise_de_algoritmos/>